# YOLO Instance Segmentation

---
## 0. Install and Mount

In [1]:
try:
    import ultralytics
except ImportError:
    !pip install ultralytics -q

# Kaggle Environment Setup
DATASET_PATH = '/kaggle/input/'  # Kaggle datasets are mounted here
OUTPUT_PATH = '/kaggle/working/'  # Kaggle output storage (persistent across sessions)

print(f"Dataset path: {DATASET_PATH}")
print(f"Output path: {OUTPUT_PATH}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 22.0 MB/s eta 0:00:00
Dataset path: /kaggle/input/
Output path: /kaggle/working/


---
## 1. Verify Dataset Location

In [2]:
import os, yaml

# Kaggle dataset path (using your specific dataset)
LOCAL_DATASET_PATH = '/kaggle/input/datasets/nguyenxuanduythai/structure-and-defect/data'

if os.path.exists(LOCAL_DATASET_PATH):
    print(f"Dataset found at: {LOCAL_DATASET_PATH}")
    print("Dataset structure:")
    for root, dirs, files in os.walk(LOCAL_DATASET_PATH):
        depth = root.replace(LOCAL_DATASET_PATH, '').count(os.sep)
        indent = '  ' * depth
        print(f"{indent}{os.path.basename(root)}/")
        if depth < 2:
            for f in files:
                print(f"{indent}  {f}")
else:
    print(f"ERROR: Dataset not found at {LOCAL_DATASET_PATH}")
    print("Make sure the dataset path is correct.")

Dataset found at: /kaggle/input/datasets/nguyenxuanduythai/structure-and-defect/data
Dataset structure:
data/
  README.roboflow.txt
  data.yaml
  valid/
    labels/
    images/
  test/
    labels/
    images/
  train/
    labels/
    images/


---
## 2. Configure data.yaml Paths

In [3]:
import shutil

data_yaml_src = os.path.join(LOCAL_DATASET_PATH, 'data.yaml')

# Copy yaml to writable working directory
data_yaml = '/kaggle/working/data.yaml'
shutil.copy(data_yaml_src, data_yaml)

with open(data_yaml, 'r') as f:
    cfg = yaml.safe_load(f)

print("Original data.yaml:")
print(yaml.dump(cfg, default_flow_style=False))

# Rewrite paths to absolute Kaggle dataset paths
cfg['path']  = LOCAL_DATASET_PATH
cfg['train'] = 'train/images'
cfg['val']   = 'valid/images'
cfg['test']  = 'test/images'

with open(data_yaml, 'w') as f:
    yaml.dump(cfg, f, default_flow_style=False)

print("Updated data.yaml paths for Kaggle:")
with open(data_yaml) as f:
    print(f.read())


Original data.yaml:
names:
- bridge
- concrete crack
- fastener corrosion
- glass crack
- irregular corrosion
- pothole
- solar panel
- tower
nc: 8
roboflow:
  license: CC BY 4.0
  project: structure-and-defect-segmentation-zul80
  url: https://app.roboflow.com/withers-workspace/structure-and-defect-segmentation-zul80/2
  version: 2
  workspace: withers-workspace
test: ../test/images
train: ../train/images
val: ../valid/images

Updated data.yaml paths for Kaggle:
names:
- bridge
- concrete crack
- fastener corrosion
- glass crack
- irregular corrosion
- pothole
- solar panel
- tower
nc: 8
path: /kaggle/input/datasets/nguyenxuanduythai/structure-and-defect/data
roboflow:
  license: CC BY 4.0
  project: structure-and-defect-segmentation-zul80
  url: https://app.roboflow.com/withers-workspace/structure-and-defect-segmentation-zul80/2
  version: 2
  workspace: withers-workspace
test: test/images
train: train/images
val: valid/images



---
## 3. Training Configuration
Edit the two dicts in ITER_CONFIGS to define your experiments.

In [4]:
# Create runs directory for saving outputs (in Kaggle's persistent storage)
RUNS_DIR = os.path.join(OUTPUT_PATH, 'training_runs')
os.makedirs(RUNS_DIR, exist_ok=True)

# Auto-detect available GPUs
import torch
AVAILABLE_GPUS = list(range(torch.cuda.device_count()))
print(f"Available GPUs: {AVAILABLE_GPUS}")

# -------------------------------------------------------
#  MODEL
# -------------------------------------------------------
# Change to a newer model string if available 
MODEL_NAME = 'yolo26s-seg.pt' 

# -------------------------------------------------------
#  COMMON SETTINGS  (shared across both iterations and are default values provided by ultralytics)
# -------------------------------------------------------
COMMON = dict(
    # ── Core Training Params ──────────────────────────────────
    data       = data_yaml,
    epochs     = 100,
    imgsz      = 640,
    project    = RUNS_DIR,
    device     = AVAILABLE_GPUS if len(AVAILABLE_GPUS) > 0 else 0,  # Use all available GPUs, or CPU
    exist_ok   = True,
    patience   = 10,
    verbose    = True,
    optimizer  = 'auto',

    # ── Color Augmentations ───────────────────────────────────
    hsv_h      = 0.015,
    hsv_s      = 0.7,
    hsv_v      = 0.4,

    # ── Geometric Augmentations ───────────────────────────────
    degrees    = 30,
    translate  = 0.1,
    scale      = 0.5,
    shear      = 0.0,
    perspective= 0.0,

    # ── Flipping ──────────────────────────────────────────────
    flipud     = 0.0, # vertical flip
    fliplr     = 0.5, # horizontal flip

    # ── Channel Augmentation ─────────────────────────────────
    bgr        = 0.0,

    # ── Advanced Augmentations ───────────────────────────────
    mosaic     = 1.0,
    mixup      = 0.0,
    cutmix     = 0.0,

    # ── Segmentation-specific ────────────────────────────────
    copy_paste      = 0.0,
    copy_paste_mode = 'flip',
    erasing      = 0.0)

# -------------------------------------------------------
#  ITERATION-SPECIFIC HYPERPARAMETERS
# -------------------------------------------------------

ITER_CONFIGS = [
    dict(
        name         = 'iter1',
        batch        = 32,   
        lrf          = 0.01,
        weight_decay = 0.0005,
        dropout      = 0.5,   # light dropout for regularisation
    ),
    #and another dict() if you want to run a second iteration with different hyperparameters
]

print("Config ready")
for c in ITER_CONFIGS:
    print(f"Iteration {c['name']}:")
    for k, v in c.items():
        if k != 'name':
            print(f"  {k}: {v}")    
            

Available GPUs: [0, 1]
Config ready
Iteration iter1:
  batch: 32
  lrf: 0.01
  weight_decay: 0.0005
  dropout: 0.5


---
## 4. Train — Iteration Loop with Auto-Resume

In [5]:
from ultralytics import YOLO
import torch

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

results_log = {}  # store result objects for later inspection

for iter_cfg in ITER_CONFIGS:
    run_name = iter_cfg['name']
    last_pt  = os.path.join(RUNS_DIR, run_name, 'weights', 'last.pt')
    best_pt  = os.path.join(RUNS_DIR, run_name, 'weights', 'best.pt')

    print(f"\n{'='*60}")
    print(f"  {run_name} ")
    print(f"{'='*60}")

    # ── AUTO-RESUME ──────────────────────────────────────────
    if os.path.exists(last_pt):
        print(f"Resuming {run_name} from checkpoint: {last_pt}")
        model = YOLO(last_pt)
        result = model.train(resume=True)

    # ── FRESH TRAINING ───────────────────────────────────────
    else:
        print(f"Fresh training — loading {MODEL_NAME}")
        model = YOLO(MODEL_NAME)

        # Merge common settings with iteration-specific overrides
        train_args = {**COMMON, **iter_cfg}
        result = model.train(**train_args)

    results_log[run_name] = result
    print(f"\n{run_name} done. Best weights saved to: {best_pt}")

print("\nAll iterations finished!")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
CUDA available: True
GPU: Tesla T4

  iter1 
Fresh training — loading yolo26s-seg.pt
Ultralytics 8.4.35 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
                                                       CUDA:1 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/data.yaml, degrees=30, deterministic=True, device=0,1, dfl=1.5, dnn=False, dropout=0.5, dynamic=False, embed=None, end2end=None, epochs=10

---
## 5. Evaluate Best Iteration on the Test Set

In [6]:
from ultralytics import YOLO
import torch

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# Change 'iter1' to another iter if available and has better val mAP
BEST_ITER = 'iter1'

best_weights = os.path.join(RUNS_DIR, BEST_ITER, 'weights', 'best.pt')
model_eval   = YOLO(best_weights)

test_metrics = model_eval.val(
    data    = data_yaml,
    split   = 'test',
    imgsz   = 640,
    verbose = True,
)

print(f"\n{'─'*40}")
print(f"Test mAP@50    (mask): {test_metrics.seg.map50:.4f}")
print(f"Test mAP@50-95 (mask): {test_metrics.seg.map:.4f}")
print(f"{'─'*40}")

CUDA available: True
GPU: Tesla T4
Ultralytics 8.4.35 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO26s-seg summary (fused): 139 layers, 10,368,436 parameters, 0 gradients, 34.1 GFLOPs
val: Fast image access ✅ (ping: 2.1±2.2 ms, read: 6.6±1.8 MB/s, size: 60.2 KB)
val: Scanning /kaggle/input/datasets/nguyenxuanduythai/structure-and-defect/data/test/labels... 1238 images, 24 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1238/1238 136.7it/s 9.1s
WARNING ⚠️ val: Cache directory /kaggle/input/datasets/nguyenxuanduythai/structure-and-defect/data/test is not writable, cache not saved.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 78/78 2.7it/s 29.3s
                   all       1238       7103      0.764      0.632      0.699      0.474      0.746      0.609      0.662      0.375
                bridge        199        307      0.839      0.812      0.863      0.566    